# 17 - Gold Customer 360

Builds exactly one row per current customer with order, net revenue, latest purchase, and most common device metrics.

This notebook is deterministic and safe to rerun. It synchronizes
the complete business result with Delta `MERGE`, records an audit
run, and refreshes aggregate-to-source lineage.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`gold`")


def sync_gold_table(target_table, source_df, keys):
    """Synchronize a deterministic Gold snapshot with an idempotent Delta MERGE."""
    source_df = source_df.dropDuplicates(keys)
    target_df = spark.table(target_table)
    stale_keys = (
        target_df.select(*keys)
        .join(source_df.select(*keys), keys, "left_anti")
        .withColumn("_delete", F.lit(True))
    )
    merge_source = (
        source_df.withColumn("_delete", F.lit(False))
        .unionByName(stale_keys, allowMissingColumns=True)
    )
    condition = " AND ".join(
        f"target.`{column}` = source.`{column}`" for column in keys
    )
    assignments = {
        column: f"source.`{column}`" for column in source_df.columns
    }
    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(merge_source.alias("source"), condition)
        .whenMatchedDelete(condition="source._delete")
        .whenMatchedUpdate(
            condition=(
                "NOT source._delete "
                "AND NOT (target.record_hash <=> source.record_hash)"
            ),
            set=assignments,
        )
        .whenNotMatchedInsert(
            condition="NOT source._delete",
            values=assignments,
        )
        .execute()
    )


def replace_gold_lineage(target_table, lineage_df):
    """Keep the current direct source mappings for one aggregate Gold table."""
    lineage_target = DeltaTable.forName(
        spark, f"{catalog}.control.record_lineage"
    )
    lineage_target.delete(F.col("target_table") == F.lit(target_table))
    prepared = (
        lineage_df.select(
            F.lit(target_table).alias("target_table"),
            F.col("target_record_key").cast("string"),
            F.lit(pipeline_run_id).alias("pipeline_run_id"),
            "source_table",
            F.col("source_record_key").cast("string"),
            "source_file",
            "batch_id",
            F.col("source_ingest_ts").cast("timestamp"),
            F.current_timestamp().alias("recorded_ts"),
        )
        .dropDuplicates(
            [
                "target_table",
                "target_record_key",
                "source_table",
                "source_record_key",
                "source_file",
                "batch_id",
            ]
        )
    )
    if prepared.limit(1).count():
        prepared.write.format("delta").mode("append").saveAsTable(
            f"{catalog}.control.record_lineage"
        )


In [ ]:
from pyspark.sql.window import Window

pipeline_name = "gold_customer_360"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
customers_table = f"{catalog}.silver.silver_customers"
orders_table = f"{catalog}.silver.silver_orders"
history_table = f"{catalog}.silver.silver_order_status_history"
events_table = f"{catalog}.silver.silver_events"
target_table = f"{catalog}.gold.gold_customer_360"
source_tables = [customers_table, orders_table, history_table, events_table]

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`gold`.`gold_customer_360` (
        user_id STRING NOT NULL,
        email STRING,
        phone STRING,
        country STRING,
        customer_created_at TIMESTAMP,
        customer_updated_at TIMESTAMP NOT NULL,
        total_orders BIGINT NOT NULL,
        total_revenue_usd DECIMAL(28,2) NOT NULL,
        latest_purchase_ts TIMESTAMP,
        most_common_device STRUCT<os: STRING, model: STRING>,
        record_hash STRING NOT NULL,
        updated_ts TIMESTAMP NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'One current customer row with order revenue and event profile'
    )
    """
)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
    "RUNNING", source_tables, job_run_id,
)

try:
    current_customers = (
        spark.table(customers_table)
        .filter(F.col("is_current"))
        .select(
            "user_id", "email", "phone", "country",
            F.col("created_at").alias("customer_created_at"),
            F.col("updated_at").alias("customer_updated_at"),
            "source_file", "batch_id", "ingest_ts",
        )
    )
    orders = spark.table(orders_table)
    order_totals = orders.groupBy("user_id").agg(
        F.countDistinct("order_id").cast("long").alias("total_orders")
    )
    history = spark.table(history_table)
    customer_revenue = (
        history.groupBy("user_id")
        .agg(
            F.sum(
                F.when(F.col("status") == "paid", F.col("subtotal_usd"))
                .when(
                    F.col("status").isin("cancelled", "returned"),
                    -F.col("subtotal_usd"),
                )
                .otherwise(F.lit(0))
            ).cast("decimal(28,2)").alias("total_revenue_usd"),
            F.max(
                F.when(F.col("status") == "paid", F.col("status_ts"))
            ).alias("latest_purchase_ts"),
        )
    )

    device_counts = (
        spark.table(events_table)
        .filter(
            F.col("user_id").isNotNull()
            & F.col("device.os").isNotNull()
            & F.col("device.model").isNotNull()
        )
        .groupBy(
            "user_id",
            F.col("device.os").alias("device_os"),
            F.col("device.model").alias("device_model"),
        )
        .count()
    )
    device_rank = Window.partitionBy("user_id").orderBy(
        F.col("count").desc(), F.col("device_os"), F.col("device_model")
    )
    top_device = (
        device_counts.withColumn("_rank", F.row_number().over(device_rank))
        .filter(F.col("_rank") == 1)
        .select(
            "user_id",
            F.struct(
                F.col("device_os").alias("os"),
                F.col("device_model").alias("model"),
            ).alias("most_common_device"),
        )
    )

    business = (
        current_customers.drop("source_file", "batch_id", "ingest_ts")
        .join(order_totals, "user_id", "left")
        .join(customer_revenue, "user_id", "left")
        .join(top_device, "user_id", "left")
        .fillna({"total_orders": 0})
        .withColumn(
            "total_revenue_usd",
            F.coalesce(
                F.col("total_revenue_usd"),
                F.lit(0).cast("decimal(28,2)"),
            ),
        )
    )
    source = (
        business.withColumn(
            "record_hash",
            F.sha2(
                F.concat_ws(
                    "||",
                    F.coalesce(F.col("email"), F.lit("")),
                    F.coalesce(F.col("phone"), F.lit("")),
                    F.coalesce(F.col("country"), F.lit("")),
                    F.coalesce(F.col("customer_created_at").cast("string"), F.lit("")),
                    F.col("customer_updated_at").cast("string"),
                    F.col("total_orders"),
                    F.col("total_revenue_usd"),
                    F.coalesce(F.col("latest_purchase_ts").cast("string"), F.lit("")),
                    F.coalesce(F.to_json("most_common_device"), F.lit("")),
                ),
                256,
            ),
        )
        .withColumn("updated_ts", F.current_timestamp())
        .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
    )
    sync_gold_table(target_table, source, ["user_id"])

    customer_lineage = current_customers.select(
        F.col("user_id").alias("target_record_key"),
        F.lit(customers_table).alias("source_table"),
        F.col("user_id").alias("source_record_key"),
        "source_file", "batch_id",
        F.col("ingest_ts").alias("source_ingest_ts"),
    )
    current_user_ids = current_customers.select("user_id")
    order_lineage = orders.filter(F.col("user_id").isNotNull()).join(
        current_user_ids, "user_id", "inner"
    ).select(
        F.col("user_id").alias("target_record_key"),
        F.lit(orders_table).alias("source_table"),
        F.col("order_id").alias("source_record_key"),
        "source_file", "batch_id",
        F.col("ingest_ts").alias("source_ingest_ts"),
    )
    history_lineage = history.filter(F.col("user_id").isNotNull()).join(
        current_user_ids, "user_id", "inner"
    ).select(
        F.col("user_id").alias("target_record_key"),
        F.lit(history_table).alias("source_table"),
        F.concat_ws("|", "order_id", "status", "status_ts")
        .alias("source_record_key"),
        "source_file", "batch_id",
        F.col("ingest_ts").alias("source_ingest_ts"),
    )
    event_lineage = (
        spark.table(events_table)
        .filter(F.col("user_id").isNotNull())
        .join(current_user_ids, "user_id", "inner")
        .select(
            F.col("user_id").alias("target_record_key"),
            F.lit(events_table).alias("source_table"),
            F.col("event_id").alias("source_record_key"),
            "source_file", "batch_id",
            F.col("ingest_ts").alias("source_ingest_ts"),
        )
    )
    replace_gold_lineage(
        target_table,
        customer_lineage.unionByName(order_lineage)
        .unionByName(history_lineage)
        .unionByName(event_lineage),
    )

    rows_read = (
        current_customers.count() + orders.count() + history.count()
        + spark.table(events_table).count()
    )
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "SUCCESS", source_tables, job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=rows_read,
        rows_written=source.count(), rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_snapshot", start_ts,
        "FAILED", source_tables, job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(target_table).orderBy(F.col("total_revenue_usd").desc()).display()
